In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("IMDB Dataset.csv")

In [4]:
df.shape

(50000, 2)

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [7]:
df.drop_duplicates(inplace=True)

In [8]:
df.shape

(49582, 2)

## Pre processing

### 1. Converting to Lower case  

In [9]:
df["review"] = df["review"].str.lower() 

### 2. Removing the URLs

In [10]:
# using Regex 
import re

In [11]:
def remove_url(text):
    text = re.sub(r"http\S+", "", text) # (pattern, repl, string)
    return text 

df["review"] = df["review"].apply(remove_url)

In [12]:
print(df["review"].apply(type).value_counts())

review
<class 'str'>    49582
Name: count, dtype: int64


### 3. Removing punctuations 

In [13]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text 
df["review"] = df["review"].apply(remove_punctuations)

In [14]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


### 4. Remove HTML 

In [15]:
def remove_html(text):
    re.sub(r"<.,*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing the stopwords

In [16]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Aditya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Aditya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aditya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [17]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords 

In [18]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text 

df["review"] = df["review"].apply(remove_stopwords)

In [19]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [20]:
# running => run 
# played => play 

from nltk.stem import PorterStemmer

In [21]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)   

df["review"] = df["review"].apply(stemming)

In [22]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [25]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["semtiment"] = le.fit_transform(df["sentiment"])

In [26]:
y = df["semtiment"]

In [27]:
y 

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: semtiment, Length: 49582, dtype: int64

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [29]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057161 stored elements and shape (49582, 5000)>

In [30]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057161 stored elements and shape (49582, 5000)>
  Coords	Values
  (0, 3537)	0.05515484681268887
  (0, 2869)	0.09361182809242374
  (0, 4940)	0.11467310366614668
  (0, 3002)	0.47200539890110405
  (0, 1276)	0.1357698919196183
  (0, 2290)	0.049500237517476293
  (0, 1934)	0.0791260308382
  (0, 3549)	0.0963974330192639
  (0, 1363)	0.06162489377343992
  (0, 1964)	0.061560444992697486
  (0, 219)	0.08588920995304898
  (0, 1621)	0.0738170550485134
  (0, 4368)	0.041994187696759305
  (0, 4170)	0.17799685402440263
  (0, 3692)	0.033532198172897175
  (0, 4737)	0.26798942924092045
  (0, 3804)	0.04427609784380831
  (0, 4769)	0.05877405881441711
  (0, 1740)	0.037520883911174724
  (0, 4496)	0.07614066339174266
  (0, 3856)	0.17537900435282314
  (0, 1631)	0.06142445471882175
  (0, 1863)	0.07433134577032253
  (0, 3329)	0.06406818508428483
  (0, 3332)	0.0844754682576354
  :	:
  (49581, 4890)	0.10682334916138103
  (49581, 1543)	0.17584072573791829

### Dataset and Dataloaders

In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [47]:
X_train.shape

(39665, 5000)

In [48]:
X_test.shape

(9917, 5000)

In [49]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [50]:
# As the data is in sparse matrix we convert it 

X_train = X_train.toarray()
X_test = X_test.toarray()

In [53]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [54]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

### Build RNN

In [57]:
import torch.nn as nn
import torch.optim as optim

In [59]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers 

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully Connected Layer 
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # OPTIONAL STEP TO SHOW DIM OF HIDDEN STATE 
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)

        out = self.fc(out[:, -1, :])
        return out 
        

In [62]:
input_size = X_train.shape[1]

model = RNN(input_size=input_size).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [64]:
epochs = 10 

for epoch in range(epochs):
    model.train()

    for Xb,yb in train_loader:

        # Move batch to GPU
        Xb = Xb.to(device)
        yb = yb.float().to(device)


        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction 

        outputs = model(Xb)

        outputs = torch.sigmoid(outputs. squeeze())

        loss = criterion(outputs, yb) 
        loss.backward()
        optimizer.step()

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/10 and loss = 0.23465538024902344
epoch = 2/10 and loss = 0.12909197807312012
epoch = 3/10 and loss = 0.2354854941368103
epoch = 4/10 and loss = 0.24434104561805725
epoch = 5/10 and loss = 0.19873741269111633
epoch = 6/10 and loss = 0.22905142605304718
epoch = 7/10 and loss = 0.2489403635263443
epoch = 8/10 and loss = 0.27481934428215027
epoch = 9/10 and loss = 0.141252800822258
epoch = 10/10 and loss = 0.19863271713256836


In [70]:
# evaluate 

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:

        # Move batch to GPU
        Xb = Xb.to(device)
        yb = yb.float().to(device)

        Xb=Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze())> 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.65090249067259
